In [ ]:
import xarray as xr
import numpy as np

In [ ]:
# ---------------------------------------------------
# User inputs
# ---------------------------------------------------



rcase

rdir0 = '/glade/scratch/rneale

file_u10  = "U10.nc"      # contains variable U10 (wind speed magnitude)
file_uv   = "UBOT_VBOT.nc"  # contains variables UBOT, VBOT

var_u10  = "U10"
var_ubot = "UBOT"
var_vbot = "VBOT"

eps = 1.0e-12  # small number to avoid division by zero

# ---------------------------------------------------
# Load data
# ---------------------------------------------------
ds_u10 = xr.open_dataset(file_u10)
ds_uv  = xr.open_dataset(file_uv)

U10  = ds_u10[var_u10]
UBOT = ds_uv[var_ubot]
VBOT = ds_uv[var_vbot]

# ---------------------------------------------------
# Ensure grids align
# ---------------------------------------------------
U10, UBOT, VBOT = xr.align(U10, UBOT, VBOT, join="exact")

# ---------------------------------------------------
# Compute wind direction unit vectors
# ---------------------------------------------------
speed_bot = np.sqrt(UBOT**2 + VBOT**2)

uhat = UBOT / (speed_bot + eps)
vhat = VBOT / (speed_bot + eps)

# ---------------------------------------------------
# Decompose U10 magnitude into components
# ---------------------------------------------------
U10_zonal      = U10 * uhat
U10_meridional = U10 * vhat

# ---------------------------------------------------
# Package output
# ---------------------------------------------------
ds_out = xr.Dataset(
    {
        "U10_zonal": U10_zonal,
        "U10_meridional": U10_meridional,
    }
)

ds_out["U10_zonal"].attrs.update({
    "long_name": "10 m wind zonal component reconstructed from magnitude",
    "units": U10.attrs.get("units", "")
})

ds_out["U10_meridional"].attrs.update({
    "long_name": "10 m wind meridional component reconstructed from magnitude",
    "units": U10.attrs.get("units", "")
})

# ---------------------------------------------------
# Write to disk
# ---------------------------------------------------
ds_out.to_netcdf("U10_decomposed_from_UBOT_VBOT.nc")

print("Decomposition complete.")